# Delivery Outcome Multiclass


In [ ]:
import sys
from pathlib import Path

# Find the project root by walking up until src/olist is visible, so the
# notebook runs from any working directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "olist").exists())
sys.path.insert(0, str(ROOT / "src"))

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

from olist import config as cfg, io_utils as io, modeling as ml, plots
from olist.features import PURCHASE_TIME_FEATURES

pd.set_option("display.max_columns", 120); pd.set_option("display.width", 220)
cfg.set_seed()
PROJECT, TARGET = "04_delivery_outcome_multiclass", "outcome_band"
paths = ml.get_paths(PROJECT)

df = io.load_ml("order_model_base")

band definitions (days vs the promised date):
  0 very early   (  -inf ,   -10]
  1 early        (   -10 ,    -3]
  2 on time      (    -3 ,     0]
  3 late         (     0 ,     7]
  4 very late    (     7 ,   inf]

                       orders  share_pct
very early              57208      57.53
early                   26698      26.85
on time                  4743       4.77
late                     4481       4.51
very late                3346       3.36
undelivered (dropped)    2965       2.98


In [ ]:
BANDS = ["very early", "early", "on time", "late", "very late"]
EDGES = [-np.inf, -10, -3, 0, 7, np.inf]
band_index = pd.cut(df["delivery_delay_days"], bins=EDGES, labels=range(5))
df[TARGET] = pd.to_numeric(band_index, errors="coerce")
for i, (lo, hi) in enumerate(zip(EDGES[:-1], EDGES[1:])):
    print(f"  {i} {BANDS[i]:<12} ({lo:>6} , {hi:>5}]")

In [ ]:
print()
dist = df[TARGET].value_counts(dropna=False).sort_index()
dist.index = [BANDS[int(i)] if pd.notna(i) else "undelivered (dropped)" for i in dist.index]
print(dist.to_frame("orders").assign(share_pct=lambda d: (d["orders"] / len(df) * 100).round(2)).to_string())

## 1. Feature contract


In [ ]:
FEATURES = [c for c in PURCHASE_TIME_FEATURES if c in df.columns]
ml.leakage_audit(FEATURES, "delivery_delay_days")
print(f"{len(FEATURES)} features, leakage audit PASS")

39 features, leakage audit PASS

split  rows  share_pct  target_mean
train 71124       73.7       0.6524
valid 12845       13.3       0.4506
 test 12507       13.0       0.8652

classes present in train: ['very early', 'early', 'on time', 'late', 'very late']


In [ ]:
data = ml.chronological_split(df, FEATURES, TARGET)
print()
print(data.summary().to_string(index=False))
labels_present = sorted(pd.unique(data.y_train.astype(int)))
print("\nclasses present in train:", [BANDS[i] for i in labels_present])

## 2. Model candidates


In [ ]:
candidates = {
    **ml.baseline_classifiers(data.y_train),
    "logreg": LogisticRegression(max_iter=3000, class_weight="balanced",
                                 random_state=cfg.RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                            class_weight="balanced_subsample",
                                            random_state=cfg.RANDOM_STATE, n_jobs=cfg.N_JOBS),
    "hist_gbm": HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06,
                                               random_state=cfg.RANDOM_STATE),
}
comparison, fitted = ml.compare_candidates(candidates, data, task="multiclass")

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,cohen_kappa,macro_roc_auc
0,random_forest,0.7190,0.3650,0.3400,0.7295,0.3484,0.3650,0.4131,0.7563
1,hist_gbm,0.8002,0.3151,0.3059,0.7631,0.4966,0.3151,0.5051,0.8081
2,logreg,0.4542,0.3857,0.2873,0.5507,0.3373,0.3857,0.2390,0.7521
3,baseline_prior,0.4880,0.2014,0.1989,0.5114,0.2012,0.2014,0.0033,0.5009
4,baseline_majority,0.6993,0.2000,0.1646,0.5755,0.1399,0.2000,0.0000,0.5000


In [ ]:
BEST_NAME = comparison.iloc[0]["model"]
best_pipe = fitted[BEST_NAME]
y_test = data.y_test.astype(int)
pred = best_pipe.predict(ml.to_sklearn_frame(data.X_test)).astype(int)
proba = best_pipe.predict_proba(ml.to_sklearn_frame(data.X_test))
model_metrics = ml.multiclass_metrics(y_test, pred, proba, labels=labels_present)

selected: random_forest
  accuracy             0.5323
  balanced_accuracy    0.3113
  macro_f1             0.2851
  weighted_f1          0.5347
  macro_precision      0.2988
  macro_recall         0.3113
  cohen_kappa          0.2982
  macro_roc_auc        0.6131

macro-F1 0.2851 vs 0.1646 for always-predict-the-biggest-band (1.7x)
Cohen's kappa 0.298 - agreement beyond chance

              precision    recall  f1-score   support

  very early       0.75      0.68      0.71      5196
       early       0.56      0.62      0.59      4909
     on time       0.12      0.03      0.04      1466
        late       0.04      0.04      0.04       764
   very late       0.02      0.20      0.04       172

    accuracy                           0.53     12507
   macro avg       0.30      0.31      0.29     12507
weighted avg       0.55      0.53      0.53     12507



In [ ]:
print("selected:", BEST_NAME)
for k, v in model_metrics.items():
    print(f"  {k:<20} {v:.4f}")

In [ ]:
majority = comparison.set_index("model").loc["baseline_majority"]
print(f"\nmacro-F1 {model_metrics['macro_f1']:.4f} vs {majority['macro_f1']:.4f} for always-predict-"
      f"the-biggest-band ({model_metrics['macro_f1'] / max(majority['macro_f1'], 1e-9):.1f}x)")
print(f"Cohen's kappa {model_metrics['cohen_kappa']:.3f} - agreement beyond chance")

In [ ]:
present = sorted(set(y_test) | set(pred))
print(classification_report(y_test, pred, labels=present,
                            target_names=[BANDS[i] for i in present], zero_division=0))

## 3. Confusion matrix


In [ ]:
cm = confusion_matrix(y_test, pred, labels=present)
distance = np.abs(np.subtract.outer(present, present))
total = cm.sum()
exact = np.trace(cm) / total
adjacent = cm[distance == 1].sum() / total
severe = cm[distance >= 2].sum() / total

print(f"exact band              : {exact:.1%}")
print(f"off by one band         : {adjacent:.1%}")
print(f"off by two or more      : {severe:.1%}   <- the errors that actually cost money")
print(f"within one band         : {exact + adjacent:.1%}")

In [ ]:
importance = ml.importance_table(best_pipe, data.X_test, y_test, top_k=12, n_repeats=3,
                                 scoring="f1_macro")

In [ ]:
saved = ml.save_artifacts(best_pipe, paths, target=TARGET, features=FEATURES,
    metrics=model_metrics,
    baseline_metrics={f"majority_{k}": float(v) for k, v in majority.items() if isinstance(v, (int, float))},
    policy={"type": "service_tier_routing", "bands": BANDS, "band_edges_days": [str(e) for e in EDGES],
            "exact_band_pct": round(exact * 100, 2), "within_one_band_pct": round((exact + adjacent) * 100, 2),
            "severe_error_pct": round(severe * 100, 2)},
    model_name=BEST_NAME,
    notes=("Replaces an earlier Seller_State target that was a deterministic function of its own "
           "features. Ordinal target: adjacent errors are cheap, non-adjacent errors are not."))

In [ ]:
print(importance.round(4).to_string(index=False))


for k, v in saved.items(): print(f"\n{k:<10} -> {v.relative_to(ROOT)}")

                feature  importance    std
estimated_delivery_days      0.1022 0.0021
       distance_km_mean     -0.0128 0.0022
        distance_bucket     -0.0127 0.0010
            distance_km     -0.0120 0.0006
         Customer_State     -0.0114 0.0023
        same_state_flag     -0.0081 0.0004
     route_prior_orders     -0.0076 0.0004
  route_prior_late_rate     -0.0053 0.0015
    order_freight_total     -0.0044 0.0012
         freight_per_kg     -0.0028 0.0001
          freight_ratio     -0.0026 0.0005
           Seller_State     -0.0025 0.0006

model      -> models\04_delivery_outcome_multiclass\best_model.joblib

metadata   -> models\04_delivery_outcome_multiclass\model_metadata.json

READ-OUT - 04_delivery_outcome_multiclass

Delivery outcome is separable into actionable service tiers at purchase time.
random_forest reaches macro-F1 0.285 (kappa 0.298)
on out-of-time orders, versus 0.165 for always predicting the largest band.
53% of orders land in exactly the right band, 77

In [ ]:
saved = ml.save_artifacts(best_pipe, paths, target=TARGET, features=FEATURES,
    metrics=model_metrics,
    baseline_metrics={f"majority_{k}": float(v) for k, v in majority.items() if isinstance(v, (int, float))},
    policy={"type": "service_tier_routing", "bands": BANDS, "band_edges_days": [str(e) for e in EDGES],
            "exact_band_pct": round(exact * 100, 2), "within_one_band_pct": round((exact + adjacent) * 100, 2),
            "severe_error_pct": round(severe * 100, 2)},
    model_name=BEST_NAME,
    notes=("Replaces an earlier Seller_State target that was a deterministic function of its own "
           "features. Ordinal target: adjacent errors are cheap, non-adjacent errors are not."))